# PyTorch 입문 — 보험 · 흡연 예측 · Sequential MLP

보험 고객이 **흡연자인지** 예/아니오로 맞춥니다.  
`get_dummies` · `StandardScaler` · `SmokeModel`(`Sequential` · LeakyReLU · Sigmoid) · `BCELoss` · Adam · `DataLoader`로 학습하고, 테스트에서 정확도·F1·precision을 봅니다.  
이론 복습: [`../session10.md`](../session10.md)

> **이 노트북이 하는 일**  
> CSV → `smoker`→target → feature(`smoker`·`target`만 제외, `charges` 포함) → train/test → get_dummies·reindex → StandardScaler(fit=train만) → 텐서 → Sequential MLP → BCELoss · Adam → 배치 학습 → 임계값 0.5 정확도·F1·precision.

환경: `torch-env` 커널을 쓰는 것을 권장합니다. `pandas` · `scikit-learn`이 필요합니다.


## 1. import


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


## 2. CSV 로드 · target · feature

경로는 노트북 기준 상대경로 `data/insurance.csv`입니다.  
`smoker`를 0/1 `target`으로 바꾸고, 학습 입력에서는 `smoker`·`target`만 뺍니다(`charges`는 남김).  
`charges`(보험료)는 흡연과 자주 같이 움직이므로, 넣으면 맞히기가 너무 쉬워질 수 있습니다. 정답을 살짝 미리 본 것과 비슷한 **쉬운 신호(누수에 가까움)** 로 이해하면 됩니다.


In [ ]:
df = pd.read_csv("data/insurance.csv")
print(df.shape)
print("결측 합계:", df.isna().sum().sum())

# yes면 1, no면 0
df["target"] = pd.get_dummies(df["smoker"])["yes"].astype("int")

features = df.drop(columns=["smoker", "target"])  # charges는 남겨 둠
label = df["target"]
print(features.shape, label.shape)


## 3. train / test 나누기

테스트는 학습에 안 쓴 점수표입니다.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, label, test_size=0.2, random_state=1
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## 4. get_dummies · reindex — 열을 train과 맞춤

`get_dummies`는 문자열 열만 골라 0/1 열로 바꿉니다.  
train과 test를 **따로** 인코딩하면, test에 안 나온 범주 열이 빠질 수 있습니다.  
그래서 test를 `reindex`로 train 열 순서에 맞추고, 없는 열은 0으로 채웁니다.


In [ ]:
X_train_encode = pd.get_dummies(X_train)
X_test_encode = pd.get_dummies(X_test)
# test에 없는 열은 0으로 채우고, 열 순서는 train과 같게
X_test_encode = X_test_encode.reindex(columns=X_train_encode.columns, fill_value=0)
print(X_train_encode.shape, X_test_encode.shape)


## 5. StandardScaler — fit은 train만

각 열을 **평균 0 · 표준편차 1**로 맞춥니다.  
테스트에 `fit`을 하면 테스트 통계가 학습 전처리에 섞입니다(데이터 누수).


In [ ]:
sc = StandardScaler()
X_train_encode = sc.fit_transform(X_train_encode)  # 학습에서만 평균·표준편차 기억
X_test_encode = sc.transform(X_test_encode)  # 테스트는 transform만


## 6. 텐서 · `unsqueeze(1)`

정답 열 수를 모델 출력 1개와 맞춥니다. `unsqueeze(1)`은 끝에 칸 1개를 붙입니다.


In [ ]:
X_train_tensor = torch.from_numpy(X_train_encode).float()
X_test_tensor = torch.from_numpy(X_test_encode).float()
y_train_tensor = torch.from_numpy(y_train.values).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test.values).float().unsqueeze(1)

print(X_train_tensor.shape, y_train_tensor.shape)
print(X_test_tensor.shape, y_test_tensor.shape)


## 7. SmokeModel — Sequential · LeakyReLU · Sigmoid

`nn.Sequential`은 층을 **위에서 아래 순서대로** 통과시킵니다.  
층을 `200→100→20→5→1`처럼 줄이며 핵심만 남깁니다.  
`LeakyReLU(0.1)`은 음수에도 작은 기울기를 남겨, 신호가 완전히 죽지 않게 합니다.  
마지막 **Sigmoid**는 출력을 0~1 확률로 만듭니다.


In [ ]:
class SmokeModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Linear(input_dim, 200),
            nn.LeakyReLU(0.1),  # 음수에도 작은 기울기
            nn.Linear(200, 100),
            nn.LeakyReLU(0.1),
            nn.Linear(100, 20),
            nn.LeakyReLU(0.1),
            nn.Linear(20, 5),
            nn.LeakyReLU(0.1),
            nn.Linear(5, output_dim),  # 칸을 줄이며 핵심만
            nn.Sigmoid(),  # 0~1 확률
        )

    def forward(self, x):
        return self.linear(x)


input_dim = X_train_tensor.shape[1]  # 인코딩 후 열 수(charges 포함 시 초안 기준 10)
output_dim = y_train_tensor.shape[1]
model = SmokeModel(input_dim, output_dim)
print(input_dim, output_dim)
print(model)

criterion = nn.BCELoss()  # 이진 정답과 확률의 차이
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(optimizer)


## 8. TensorDataset · DataLoader

같은 사람의 특징·흡연 여부를 한 묶음으로 맞춥니다.  
한 번에 128개씩, epoch마다 순서를 섞습니다.


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print(len(train_dataset), len(train_loader))


## 9. 학습 루프

배치마다 `zero_grad` → `backward` → `step` 순서를 지킵니다.  
epoch마다 평균 loss를 모아 곡선을 그립니다.


In [ ]:
epochs = 100
train_losses = []  # epoch별 평균 loss

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        yhat = model(x_batch)
        loss = criterion(yhat, y_batch)

        optimizer.zero_grad()  # 이전 기울기 비우기
        loss.backward()
        optimizer.step()  # weight · bias 갱신

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:3d} | train_bce={avg_loss:.4f}")


## 10. loss 곡선

가로=epoch, 세로=평균 BCE입니다. 내려가면 학습이 차이를 줄이고 있다는 뜻입니다.


In [ ]:
plt.plot(train_losses)
plt.xlabel("epoch")
plt.ylabel("train BCE")
plt.title("학습 loss 곡선")
plt.show()


## 11. 테스트 — 임계값 0.5 · 정확도 · F1 · precision

예측 확률이 **0.5 초과**이면 흡연(1), 아니면 비흡연(0)으로 봅니다.  
정확도는 "맞춘 개수 / 전체"입니다.  
`f1_score` · `precision_score`는 넘파이 배열 기준으로 만들어져 있어, 텐서를 `.detach().numpy()`로 바꾼 뒤 호출합니다.

성능이 아쉬울 때 짧게 볼 점:
- 입력 열(`charges` 포함 여부)을 다시 확인
- 임계값을 0.5에서 조금 낮추거나 올리기
- 은닉층을 줄여 모델 단순화
- 로지스틱 회귀와 비교


In [ ]:
from sklearn.metrics import f1_score, precision_score

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model.eval()
total = 0
correct = 0
y_true_list = []
y_pred_list = []

with torch.no_grad():  # 평가에서는 기울기 계산 끔
    for x_batch, y_batch in test_loader:
        y_prob = model(x_batch)
        y_pred = (y_prob > 0.5).float()  # 임계 0.5
        total += y_batch.size(0)
        correct += (y_pred == y_batch).sum().item()
        y_true_list.append(y_batch)
        y_pred_list.append(y_pred)

acc = correct / total
# sklearn은 넘파이 배열을 받으므로 텐서를 넘파이로 바꿈
y_true_np = torch.cat(y_true_list).detach().numpy()
y_pred_np = torch.cat(y_pred_list).detach().numpy()

print("테스트 전체:", total)
print("맞춘 개수:", correct)
print("테스트 정확도:", acc)
print("F1:", f1_score(y_true_np, y_pred_np))
print("precision:", precision_score(y_true_np, y_pred_np))


---

이론 복습: [`../session10.md`](../session10.md) · 이전 실습: [`11_churn_logistic.ipynb`](./11_churn_logistic.ipynb)
